<a href="https://colab.research.google.com/github/Prakum14/MLOps/blob/main/Distributed_Matrix_Multiplication_using_MPI_(Python).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install MPI
!pip install mpi4py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.5/106.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.3/466.3 kB 17.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mpi4py: filename=mpi4py-4.0.3-cp311-cp311-linux_x86_64.whl size=4442048 sha256=8e83511f0f83e0190dc78fba11651f220798d1c6ba938f08d3a184acca6763e7
  Stored in directory: /root/.cache/pip/wheels/5c/56/17/bf6ba37aa971a191a8b9eaa188bf5ec855b8911c1c56fb1f84
Successfully built mpi4py


In [4]:
# Import necessary libraries
import pandas as pd
import numpy as np

from mpi4py import MPI
import time
import os



In [9]:
# --- Configuration for Matrix Multiplication ---
# These values will define the dimensions of the matrices.
# For simplicity, we'll use square matrices for this implementation.
# N x M matrix multiplied by M x P matrix. Here, N=M=P=MATRIX_DIM
MATRIX_DIM = 500 # Default dimension for square matrices. Can be changed for testing.



In [5]:
# --- MPI Initialization ---
# MPI.COMM_WORLD is the default communicator that includes all processes.
comm = MPI.COMM_WORLD
rank = comm.Get_rank() # Get the unique ID (rank) of the current process
size = comm.Get_size() # Get the total number of processes in the communicator

# --- Helper Function to Create Random Matrices ---
def create_random_matrix(rows, cols, seed=None):
    """
    Creates a random matrix of specified dimensions.
    Args:
        rows (int): Number of rows.
        cols (int): Number of columns.
        seed (int, optional): Seed for random number generation for reproducibility.
    Returns:
        np.array: A NumPy array representing the matrix.
    """
    if seed is not None:
        np.random.seed(seed)
    return np.random.rand(rows, cols) * 10 # Values between 0 and 10



In [6]:
# --- Serial Matrix Multiplication Implementation ---
def serial_matrix_multiplication(A, B):
    """
    Performs standard serial matrix multiplication C = A @ B.
    Args:
        A (np.array): First matrix.
        B (np.array): Second matrix.
    Returns:
        np.array: Resultant matrix C.
    """
    # Check for valid dimensions for multiplication
    if A.shape[1] != B.shape[0]:
        raise ValueError("Matrix dimensions are not compatible for multiplication.")

    print(f"[{os.getpid()}] Performing serial matrix multiplication on {A.shape[0]}x{A.shape[1]} and {B.shape[0]}x{B.shape[1]} matrices...")
    start_time = time.time()
    C = np.dot(A, B) # Using NumPy's highly optimized dot product
    end_time = time.time()
    execution_time = end_time - start_time
    print(f"[{os.getpid()}] Serial multiplication completed in {execution_time:.4f} seconds.")
    return C, execution_time



In [7]:
# --- Distributed Matrix Multiplication Implementation (Block Row Partitioning) ---
def distributed_matrix_multiplication(MATRIX_DIM, comm, rank, size):
    """
    Performs distributed matrix multiplication using MPI with block row partitioning.
    Matrix A is partitioned by rows and distributed among processes.
    Matrix B is broadcast to all processes.
    Each process computes its part of C, and results are gathered at the root.

    Args:
        MATRIX_DIM (int): Dimension of the square matrices.
        comm (MPI.Comm): MPI communicator.
        rank (int): Rank of the current process.
        size (int): Total number of processes.

    Returns:
        np.array: Resultant matrix C (only at root process).
        float: Execution time of the distributed operation (only at root process).
    """
    A = None
    B = None
    local_A = None
    C = None
    start_time = 0

    if rank == 0:
        # Root process creates the full matrices
        A = create_random_matrix(MATRIX_DIM, MATRIX_DIM, seed=0)
        B = create_random_matrix(MATRIX_DIM, MATRIX_DIM, seed=1)
        print(f"\n[Rank {rank}/{size}] Starting distributed matrix multiplication for {MATRIX_DIM}x{MATRIX_DIM} matrices.")
        start_time = time.time()

    # Broadcast Matrix B to all processes
    # All processes need the full Matrix B to compute their local part of C
    B = comm.bcast(B, root=0)

    # Determine the chunk of rows for Matrix A for each process
    # This is a simple block distribution
    rows_per_process = MATRIX_DIM // size
    # Calculate indices for slicing A
    start_row = rank * rows_per_process
    end_row = (rank + 1) * rows_per_process if rank < size - 1 else MATRIX_DIM

    # Send relevant chunk of A to each process (including root itself)
    # Using scatter instead of manually slicing and sending/receiving
    # For scatter, A needs to be sliced into pieces beforehand for efficiency
    # If A is too large for scatter, manual send/recv for slices is an alternative.
    # Here, we will manually slice and send/receive or use scatter for fixed size.

    # Simpler approach: Root slices and sends, others receive their slice
    if rank == 0:
        # Split A into chunks for scattering
        # Ensure that remaining rows go to the last process
        A_chunks = [A[i * rows_per_process : (i + 1) * rows_per_process] for i in range(size - 1)]
        A_chunks.append(A[(size - 1) * rows_per_process :]) # Last chunk gets remaining rows
        print(f"[Rank {rank}/{size}] Matrix A partitioned into {len(A_chunks)} chunks.")
    else:
        A_chunks = None # Non-root processes don't need this

    # Scatter the chunks of A to all processes
    local_A = comm.scatter(A_chunks, root=0)
    print(f"[Rank {rank}/{size}] Received local_A of shape: {local_A.shape}")

    # Perform local matrix multiplication: C_local = local_A @ B
    local_C = np.dot(local_A, B)
    print(f"[Rank {rank}/{size}] Completed local computation.")

    # Gather all local_C parts back to the root process
    # The root process will receive a list of arrays, which need to be concatenated
    gathered_C_parts = comm.gather(local_C, root=0)

    total_execution_time = 0
    if rank == 0:
        # Concatenate the gathered parts to form the full result matrix C
        C = np.vstack(gathered_C_parts)
        end_time = time.time()
        total_execution_time = end_time - start_time
        print(f"[Rank {rank}/{size}] Distributed multiplication completed in {total_execution_time:.4f} seconds.")
        # Optional: Verify dimensions
        # print(f"[Rank {rank}/{size}] Final C shape: {C.shape}")

    return C, total_execution_time



In [10]:
# --- Main Execution Block ---
if __name__ == "__main__":
    matrix_dim = MATRIX_DIM # Use the configured dimension

    # --- Task: Environment Setup (Instructions below) ---
    # --- Task: Matrix Multiplication (Serial Implementation) ---
    # Deliverable: serial_matrix_multiplication function above.
    # It will run if rank == 0.

    # --- Task: Distributed Implementation (MPI) ---
    # Deliverable: distributed_matrix_multiplication function above.

    # --- Task: Performance Metrics & Benchmarking ---
    # Deliverable: The main execution logic below will collect and print metrics.

    # Only rank 0 handles the overall execution and comparison
    if rank == 0:
        print("Starting Matrix Multiplication Project...\n")

        # --- Run Serial Benchmark ---
        print("\n--- Running Serial Matrix Multiplication ---")
        A_serial = create_random_matrix(matrix_dim, matrix_dim, seed=0)
        B_serial = create_random_matrix(matrix_dim, matrix_dim, seed=1)
        _, serial_time = serial_matrix_multiplication(A_serial, B_serial)

        # --- Run Distributed Benchmark ---
        print("\n--- Running Distributed Matrix Multiplication ---")
        # Ensure matrix dimension is divisible by number of processes for simple block partitioning
        if matrix_dim % size != 0:
            print(f"Warning: MATRIX_DIM ({matrix_dim}) is not perfectly divisible by number of processes ({size}).")
            print("The last process will handle the remaining rows. This is handled by the vstack logic in gather.")

        distributed_C, distributed_time = distributed_matrix_multiplication(matrix_dim, comm, rank, size)

        # --- Verify Results (Optional, for smaller matrices) ---
        # For very large matrices, uncommenting this might consume a lot of memory
        # if distributed_C is not None and A_serial is not None and B_serial is not None:
        #     serial_C, _ = serial_matrix_multiplication(A_serial, B_serial) # Re-run serial for comparison
        #     if np.allclose(serial_C, distributed_C):
        #         print("\nResult verification: Distributed result matches serial result (within tolerance).")
        #     else:
        #         print("\nResult verification: WARNING - Distributed result DOES NOT match serial result!")

        # --- Report Performance Metrics ---
        print("\n" + "="*60)
        print("               PERFORMANCE REPORT                 ")
        print("="*60)
        print(f"Matrix Dimension: {matrix_dim}x{matrix_dim}")
        print(f"Number of MPI Processes: {size}")
        print(f"Serial Execution Time: {serial_time:.4f} seconds")
        print(f"Distributed Execution Time: {distributed_time:.4f} seconds")
        if distributed_time > 0: # Avoid division by zero if distributed_time is tiny
            speedup = serial_time / distributed_time
            print(f"Speedup (Serial / Distributed): {speedup:.2f}x")
            print(f"Efficiency (Speedup / Processes): {speedup / size:.2f}")
        else:
            print("Distributed time was too low to calculate speedup/efficiency.")
        print("="*60)

    # All processes synchronize before exiting
    comm.Barrier()

Starting Matrix Multiplication Project...


--- Running Serial Matrix Multiplication ---
[767] Performing serial matrix multiplication on 5000x5000 and 5000x5000 matrices...
[767] Serial multiplication completed in 5.0716 seconds.

--- Running Distributed Matrix Multiplication ---

[Rank 0/1] Starting distributed matrix multiplication for 5000x5000 matrices.
[Rank 0/1] Matrix A partitioned into 1 chunks.
[Rank 0/1] Received local_A of shape: (5000, 5000)
[Rank 0/1] Completed local computation.
[Rank 0/1] Distributed multiplication completed in 5.1228 seconds.

               PERFORMANCE REPORT                 
Matrix Dimension: 5000x5000
Number of MPI Processes: 1
Serial Execution Time: 5.0716 seconds
Distributed Execution Time: 5.1228 seconds
Speedup (Serial / Distributed): 0.99x
Efficiency (Speedup / Processes): 0.99
